# Validation SQL des KPIs — DuckDB
## Contexte
Ce notebook valide les KPIs calculés dans l'EDA Python en les recalculant via des requêtes SQL avec DuckDB.

**Périmètre :** France / Women / 2023-2024  
**Statuts :** Complete (ventes) · Returned (retours)  
**Source :** CSV local chargé via DuckDB

In [1]:
# Importation et connexion DuckDB
import duckdb
import pandas as pd

# Connexion DuckDB et chargement du CSV
con = duckdb.connect()

# Vérification du chargement
result = con.execute("""
    SELECT COUNT(*) as nb_lignes, 
           COUNT(DISTINCT order_id) as nb_commandes
    FROM '../data/thelook_fr_women_2023_2024.csv'
    WHERE item_status = 'Complete'
""").df()

print("✅ DuckDB connecté et CSV chargé")
print(result)

✅ DuckDB connecté et CSV chargé
   nb_lignes  nb_commandes
0        425           281


## KPI 1 — Chiffre d'affaires

In [4]:
# KPI 1 — Chiffre d'affaires par année
ca_sql = con.execute("""
    SELECT
        YEAR(CAST(item_created_at AS TIMESTAMP)) AS annee,
        ROUND(SUM(sale_price), 2)                AS ca_total,
        COUNT(*)                                  AS nb_lignes
    FROM '../data/thelook_fr_women_2023_2024.csv'
    WHERE item_status = 'Complete'
    GROUP BY annee
    ORDER BY annee
""").df()

print("=== KPI 1 — Chiffre d'affaires ===")
print(ca_sql)
print(f"\n✅ Validation Python : 2023 = 7 806€ / 2024 = 15 716€")

=== KPI 1 — Chiffre d'affaires ===
   annee  ca_total  nb_lignes
0   2023   7806.32        146
1   2024  15716.28        279

✅ Validation Python : 2023 = 7 806€ / 2024 = 15 716€


In [5]:
# KPI 2 — Marge brute par année
marge_sql = con.execute("""
    SELECT
        YEAR(CAST(item_created_at AS TIMESTAMP)) AS annee,
        ROUND(SUM(sale_price - cost), 2)          AS marge_brute
    FROM '../data/thelook_fr_women_2023_2024.csv'
    WHERE item_status = 'Complete'
    GROUP BY annee
    ORDER BY annee
""").df()

# KPI 3 — Panier moyen par année
panier_sql = con.execute("""
    SELECT
        YEAR(CAST(item_created_at AS TIMESTAMP))        AS annee,
        ROUND(SUM(sale_price) / COUNT(DISTINCT order_id), 2) AS panier_moyen
    FROM '../data/thelook_fr_women_2023_2024.csv'
    WHERE item_status = 'Complete'
    AND sale_price > 0
    GROUP BY annee
    ORDER BY annee
""").df()

# KPI 4 — Taux de retour par année
taux_retour_sql = con.execute("""
    SELECT
        YEAR(CAST(item_created_at AS TIMESTAMP)) AS annee,
        ROUND(
            COUNT(CASE WHEN item_status = 'Returned' THEN 1 END) * 100.0 /
            COUNT(CASE WHEN item_status IN ('Complete', 'Returned') THEN 1 END),
        1) AS taux_retour_pct
    FROM '../data/thelook_fr_women_2023_2024.csv'
    WHERE item_status IN ('Complete', 'Returned')
    GROUP BY annee
    ORDER BY annee
""").df()

print("=== KPI 2 — Marge brute ===")
print(marge_sql)
print(f"\n✅ Validation Python : 2023 = 4 075€ / 2024 = 8 135€")

print("\n=== KPI 3 — Panier moyen ===")
print(panier_sql)
print(f"\n✅ Validation Python : 2023 = 80.48€ / 2024 = 85.41€")

print("\n=== KPI 4 — Taux de retour ===")
print(taux_retour_sql)
print(f"\n✅ Validation Python : 2023 = 37.9% / 2024 = 30.4%")

=== KPI 2 — Marge brute ===
   annee  marge_brute
0   2023      4075.34
1   2024      8134.61

✅ Validation Python : 2023 = 4 075€ / 2024 = 8 135€

=== KPI 3 — Panier moyen ===
   annee  panier_moyen
0   2023         80.48
1   2024         85.41

✅ Validation Python : 2023 = 80.48€ / 2024 = 85.41€

=== KPI 4 — Taux de retour ===
   annee  taux_retour_pct
0   2023             37.9
1   2024             30.4

✅ Validation Python : 2023 = 37.9% / 2024 = 30.4%


In [6]:
# KPI 5 — Taux de réachat par année
reachat_sql = con.execute("""
    WITH commandes_par_client AS (
        SELECT
            YEAR(CAST(item_created_at AS TIMESTAMP)) AS annee,
            user_id,
            COUNT(DISTINCT order_id) AS nb_commandes
        FROM '../data/thelook_fr_women_2023_2024.csv'
        WHERE item_status = 'Complete'
        GROUP BY annee, user_id
    )
    SELECT
        annee,
        ROUND(
            COUNT(CASE WHEN nb_commandes >= 2 THEN 1 END) * 100.0 /
            COUNT(*),
        1) AS taux_reachat_pct
    FROM commandes_par_client
    GROUP BY annee
    ORDER BY annee
""").df()

print("=== KPI 5 — Taux de réachat ===")
print(reachat_sql)
print(f"\n✅ Validation Python : 2023 = 0.0% / 2024 = 3.4%")

=== KPI 5 — Taux de réachat ===
   annee  taux_reachat_pct
0   2023               0.0
1   2024               3.4

✅ Validation Python : 2023 = 0.0% / 2024 = 3.4%


## Synthèse de la validation SQL

| KPI | Python | SQL | Écart | Statut |
|-----|--------|-----|-------|--------|
| CA 2023 | 7 806€ | 7 806.32€ | ~0€ | ✅ Validé |
| CA 2024 | 15 716€ | 15 716.28€ | ~0€ | ✅ Validé |
| Marge 2023 | 4 075€ | 4 075.34€ | ~0€ | ✅ Validé |
| Marge 2024 | 8 135€ | 8 134.61€ | ~0€ | ✅ Validé |
| Panier moyen 2023 | 80.48€ | 80.48€ | 0€ | ✅ Validé |
| Panier moyen 2024 | 85.41€ | 85.41€ | 0€ | ✅ Validé |
| Taux retour 2023 | 37.9% | 37.9% | 0pp | ✅ Validé |
| Taux retour 2024 | 30.4% | 30.4% | 0pp | ✅ Validé |
| Taux réachat 2023 | 0.0% | 0.0% | 0pp | ✅ Validé |
| Taux réachat 2024 | 3.4% | 3.4% | 0pp | ✅ Validé |

**Conclusion :** Les légères différences (arrondis) entre Python et SQL sont dues aux arrondis d'affichage — les calculs sont rigoureusement cohérents.